# MEGA-RAG: Evaluate with Local Model on Colab GPU

Runs PubMedQA evaluation using a model loaded directly on Colab's T4 GPU.
**No API calls, no rate limits, no token quotas.**

## Setup
1. Runtime → T4 GPU
2. Run all cells (~15-20 min for 50 samples)

In [ ]:
!pip install -q transformers bitsandbytes accelerate torch
print('Done')

In [ ]:
import os, sys

# Clone project if needed
if not os.path.exists('/content/medicalq-a_rag'):
    !git clone https://github.com/Dev07-Harsh/medicalq-a_rag.git /content/medicalq-a_rag
else:
    !cd /content/medicalq-a_rag && git pull

os.chdir('/content/medicalq-a_rag')
sys.path.insert(0, '/content/medicalq-a_rag')
print(f'Dir: {os.getcwd()}')

In [ ]:
# Load model directly on GPU — no API needed!
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'  # Same model being fine-tuned on Kaggle

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

print(f'Loading {MODEL_ID} in 4-bit on GPU...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map='auto', torch_dtype=compute_dtype
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Loaded! GPU memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB / 15 GB')

In [ ]:
# Load test data
import json, re, time
from collections import Counter

with open('pubmedQA/splits/test.json') as f:
    test_data = json.load(f)

# Sample 50 for quick test
import random
random.seed(42)
test_ids = random.sample(list(test_data.keys()), 50)
samples = [(pid, test_data[pid]) for pid in test_ids]
print(f'Test samples: {len(samples)}')

gt_dist = Counter(test_data[pid].get('final_decision', '').lower() for pid, _ in samples)
print(f'Distribution: {dict(gt_dist)}')

In [ ]:
def predict(question, contexts=None, max_tokens=256):
    """Run inference on GPU — no API call."""
    if contexts:
        evidence = '\n\n'.join(f'[Evidence {i+1}] {c}' for i, c in enumerate(contexts))
        user_msg = (
            f'Based ONLY on the evidence, answer yes/no/maybe.\n\n'
            f'EVIDENCE:\n{evidence}\n\n'
            f'QUESTION: {question}\n\n'
            f'Brief reasoning, then: Final Answer: yes|no|maybe'
        )
    else:
        user_msg = (
            f'Answer this medical research question with yes, no, or maybe.\n\n'
            f'QUESTION: {question}\n\n'
            f'Brief reasoning, then: Final Answer: yes|no|maybe'
        )
    
    messages = [
        {'role': 'system', 'content': 'You are a medical expert. Answer with brief reasoning then Final Answer: yes/no/maybe.'},
        {'role': 'user', 'content': user_msg},
    ]
    
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=1024).to(model.device)
    
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens, temperature=0.3, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)


def extract_decision(text):
    t = (text or '').lower().strip()
    m = re.search(r'final\s*answer\s*[:\s]*(yes|no|maybe)', t)
    if m: return m.group(1)
    for w in ['yes', 'no', 'maybe']:
        if t.startswith(w): return w
    return 'unknown'


# Quick test
pid, item = samples[0]
resp = predict(item['QUESTION'], item.get('CONTEXTS'))
print(f'Q: {item["QUESTION"][:80]}...')
print(f'Response: {resp[:200]}')
print(f'Prediction: {extract_decision(resp)}')
print(f'Ground truth: {item["final_decision"]}')

In [ ]:
# Run evaluation: llm_only (no context) and oracle_context (gold contexts)
from tqdm.auto import tqdm

results = {'llm_only': {'correct': 0, 'total': 0, 'preds': [], 'gts': []},
           'oracle_context': {'correct': 0, 'total': 0, 'preds': [], 'gts': []}}

for i, (pid, item) in enumerate(tqdm(samples, desc='Evaluating')):
    q = item['QUESTION']
    gt = item.get('final_decision', '').lower()
    contexts = item.get('CONTEXTS', [])
    if gt not in {'yes', 'no', 'maybe'}: continue
    
    # LLM only (no context)
    resp1 = predict(q)
    pred1 = extract_decision(resp1)
    results['llm_only']['preds'].append(pred1)
    results['llm_only']['gts'].append(gt)
    results['llm_only']['total'] += 1
    if pred1 == gt: results['llm_only']['correct'] += 1
    
    # Oracle context (gold evidence provided)
    resp2 = predict(q, contexts)
    pred2 = extract_decision(resp2)
    results['oracle_context']['preds'].append(pred2)
    results['oracle_context']['gts'].append(gt)
    results['oracle_context']['total'] += 1
    if pred2 == gt: results['oracle_context']['correct'] += 1
    
    if (i+1) % 10 == 0:
        acc1 = results['llm_only']['correct'] / results['llm_only']['total']
        acc2 = results['oracle_context']['correct'] / results['oracle_context']['total']
        print(f'  [{i+1}/{len(samples)}] llm_only={acc1:.1%} oracle={acc2:.1%}')

In [ ]:
# Save results
import json
os.makedirs('evaluation_results', exist_ok=True)
output = {
    'model': MODEL_ID,
    'fine_tuned': False,
    'samples': len(samples),
    'results': {k: {'accuracy': v['correct']/v['total'] if v['total'] else 0,
                     'correct': v['correct'], 'total': v['total'],
                     'predictions': dict(Counter(v['preds']))}
                for k, v in results.items()}
}
with open('evaluation_results/eval_qwen7b_base.json', 'w') as f:
    json.dump(output, f, indent=2)
print('Saved to evaluation_results/eval_qwen7b_base.json')

In [ ]:
# Save results
import json
os.makedirs('evaluation_results', exist_ok=True)
output = {
    'model': MODEL_ID,
    'fine_tuned': False,
    'samples': len(samples),
    'results': {k: {'accuracy': v['correct']/v['total'] if v['total'] else 0,
                     'correct': v['correct'], 'total': v['total'],
                     'predictions': dict(Counter(v['preds']))}
                for k, v in results.items()}
}
with open('evaluation_results/eval_qwen3b_base.json', 'w') as f:
    json.dump(output, f, indent=2)
print('Saved to evaluation_results/eval_qwen3b_base.json')